### Import Dependencies

In [1]:
import pandas as pd
import csv
import json
import requests
import datetime as dt
import db_connections
import config
import time
import os

/Users/nehlaamin/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


Pinged your deployment. You successfully connected to MongoDB!


### Extract CSV and API Data

In [2]:
# Extract Indego Q2 data into dataframe

# Load dataset
file_path = "Dataset/indego-trips-2025-q2.csv"

indego_df = pd.read_csv(file_path, low_memory=False)

# Preview dataframe
indego_df.head()


,trip_id,duration,start_time,end_time,start_station,start_lat,start_lon,end_station,end_lat,end_lon,bike_id,plan_duration,trip_route_category,passholder_type,bike_type
0,1164246461,11,4/1/2025 0:04,4/1/2025 0:15,3022,39.954720,-75.183228,3064,39.938400,-75.173271,31751,30,One Way,Indego30,electric
1,1164246634,31,4/1/2025 0:04,4/1/2025 0:35,3040,39.962891,-75.166061,3100,39.927769,-75.151031,14481,30,One Way,Indego30,standard
2,1164246553,9,4/1/2025 0:17,4/1/2025 0:26,3396,39.923271,-75.182098,3349,39.936508,-75.186211,02724,30,One Way,Indego30,standard
3,1164246521,3,4/1/2025 0:20,4/1/2025 0:23,3054,39.962502,-75.174202,3235,39.959999,-75.165100,24841,365,One Way,Indego365,electric
4,1164246659,11,4/1/2025 0:25,4/1/2025 0:36,3280,39.939678,-75.213623,3349,39.936508,-75.186211,25744,30,One Way,Indego30,electric


In [3]:
# OpenWeather API setup
API_KEY = config.weather_api_key

# Base URL
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

# Example coordinates (Philadelphia center)
lat = 39.9526
lon = -75.1652

# Test API call
params = {
    "lat": lat,
    "lon": lon,
    "appid": API_KEY,
    "units": "imperial"  # Fahrenheit
}

response = requests.get(BASE_URL, params=params)

# Check response
if response.status_code == 200:
    weather_data = response.json()
    print("API Connection Successful")
    print(weather_data["weather"][0]["description"])
else:
    print("API Error:", response.status_code)

API Connection Successful
overcast clouds


In [4]:
print(response.json())

{'coord': {'lon': -75.1652, 'lat': 39.9526}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04n'}], 'base': 'stations', 'main': {'temp': 58.69, 'feels_like': 58.12, 'temp_min': 58.69, 'temp_max': 58.69, 'pressure': 1012, 'humidity': 82, 'sea_level': 1012, 'grnd_level': 1007}, 'visibility': 10000, 'wind': {'speed': 7.16, 'deg': 154, 'gust': 16.87}, 'clouds': {'all': 100}, 'dt': 1776570292, 'sys': {'country': 'US', 'sunrise': 1776507496, 'sunset': 1776555678}, 'timezone': -14400, 'id': 4560349, 'name': 'Philadelphia', 'cod': 200}


In [5]:
data = response.json()

temp_f = data.get("main", {}).get("temp")
temp_c = ((temp_f - 32) * 5 / 9) if temp_f is not None else None
            
wind_speed = data.get("wind", {}).get("speed")

humidity = data.get("main", {}).get("humidity")

print(f'Temp F: {temp_f}\nTemp C: {temp_c}\nWind Speed: {wind_speed}\nHumidity: {humidity}')

Temp F: 58.69
Temp C: 14.827777777777776
Wind Speed: 7.16
Humidity: 82


### Transform Indego dataframe data

In [6]:
# Indego Dataframe transformation
# Replace missing values for rows where start_station == 3000
indego_df.loc[
    (indego_df["start_station"] == 3000) & (indego_df["start_lat"].isna()),
    "start_lat"
] = 39.954613

indego_df.loc[
    (indego_df["start_station"] == 3000) & (indego_df["start_lon"].isna()),
    "start_lon"
] = -75.188439


# Replace missing values for rows where end_station == 3000
indego_df.loc[
    (indego_df["end_station"] == 3000) & (indego_df["end_lat"].isna()),
    "end_lat"
] = 39.954613

indego_df.loc[
    (indego_df["end_station"] == 3000) & (indego_df["end_lon"].isna()),
    "end_lon"
] = -75.188439

In [7]:
# Check remaining missing values
indego_df[["start_lat", "start_lon", "end_lat", "end_lon"]].isna().sum()

start_lat    0
start_lon    0
end_lat      0
end_lon      0
dtype: int64

In [8]:
# Convert start_time and end_time column data types to datetime
indego_df["start_time"] = pd.to_datetime(indego_df["start_time"], errors="coerce")
indego_df["end_time"] = pd.to_datetime(indego_df["end_time"], errors="coerce")

# Create new features
indego_df["start_date"] = indego_df["start_time"].dt.date
indego_df["start_time_only"] = indego_df["start_time"].dt.time
indego_df["end_date"] = indego_df["end_time"].dt.date
indego_df["end_time_only"] = indego_df["end_time"].dt.time

# Validate changes
indego_df.head()

,trip_id,duration,start_time,end_time,start_station,start_lat,start_lon,end_station,end_lat,end_lon,bike_id,plan_duration,trip_route_category,passholder_type,bike_type,start_date,start_time_only,end_date,end_time_only
0,1164246461,11,2025-04-01 00:04:00,2025-04-01 00:15:00,3022,39.954720,-75.183228,3064,39.938400,-75.173271,31751,30,One Way,Indego30,electric,2025-04-01,00:04:00,2025-04-01,00:15:00
1,1164246634,31,2025-04-01 00:04:00,2025-04-01 00:35:00,3040,39.962891,-75.166061,3100,39.927769,-75.151031,14481,30,One Way,Indego30,standard,2025-04-01,00:04:00,2025-04-01,00:35:00
2,1164246553,9,2025-04-01 00:17:00,2025-04-01 00:26:00,3396,39.923271,-75.182098,3349,39.936508,-75.186211,02724,30,One Way,Indego30,standard,2025-04-01,00:17:00,2025-04-01,00:26:00
3,1164246521,3,2025-04-01 00:20:00,2025-04-01 00:23:00,3054,39.962502,-75.174202,3235,39.959999,-75.165100,24841,365,One Way,Indego365,electric,2025-04-01,00:20:00,2025-04-01,00:23:00
4,1164246659,11,2025-04-01 00:25:00,2025-04-01 00:36:00,3280,39.939678,-75.213623,3349,39.936508,-75.186211,25744,30,One Way,Indego30,electric,2025-04-01,00:25:00,2025-04-01,00:36:00


In [9]:
# Drop the original start_time and end_time columns
indego_df = indego_df.drop(columns=["start_time", "end_time"])

# Validate changes
indego_df.columns

Index(['trip_id', 'duration', 'start_station', 'start_lat', 'start_lon',
       'end_station', 'end_lat', 'end_lon', 'bike_id', 'plan_duration',
       'trip_route_category', 'passholder_type', 'bike_type', 'start_date',
       'start_time_only', 'end_date', 'end_time_only'],
      dtype='object')

### Create Weather Dataframe using OpenWeather API

In [10]:
# Make a weather dataframe that lists the wind speed, precipitation, temperature in Farenheight 
# and temperature in Celsius depending on the particular geographic coordinates and date

# Round coordinates to reduce tiny decimal differences
indego_df["start_lat_round"] = indego_df["start_lat"].round(3)
indego_df["start_lon_round"] = indego_df["start_lon"].round(3)

# Validate changes
indego_df.head()

,trip_id,duration,start_station,start_lat,start_lon,end_station,end_lat,end_lon,bike_id,plan_duration,trip_route_category,passholder_type,bike_type,start_date,start_time_only,end_date,end_time_only,start_lat_round,start_lon_round
0,1164246461,11,3022,39.954720,-75.183228,3064,39.938400,-75.173271,31751,30,One Way,Indego30,electric,2025-04-01,00:04:00,2025-04-01,00:15:00,39.955,-75.183
1,1164246634,31,3040,39.962891,-75.166061,3100,39.927769,-75.151031,14481,30,One Way,Indego30,standard,2025-04-01,00:04:00,2025-04-01,00:35:00,39.963,-75.166
2,1164246553,9,3396,39.923271,-75.182098,3349,39.936508,-75.186211,02724,30,One Way,Indego30,standard,2025-04-01,00:17:00,2025-04-01,00:26:00,39.923,-75.182
3,1164246521,3,3054,39.962502,-75.174202,3235,39.959999,-75.165100,24841,365,One Way,Indego365,electric,2025-04-01,00:20:00,2025-04-01,00:23:00,39.963,-75.174
4,1164246659,11,3280,39.939678,-75.213623,3349,39.936508,-75.186211,25744,30,One Way,Indego30,electric,2025-04-01,00:25:00,2025-04-01,00:36:00,39.940,-75.214


In [11]:
# Unique keys for weather lookup
weather_keys = (
    indego_df[["start_lat_round", "start_lon_round", "start_date"]]
    .dropna()
    .drop_duplicates()
    .rename(columns={
        "start_lat_round": "start_lat",
        "start_lon_round": "start_lon"
    })
    .reset_index(drop=True)
)

print(weather_keys.shape)
weather_keys.head()

(23102, 3)


,start_lat,start_lon,start_date
0,39.955,-75.183,2025-04-01
1,39.963,-75.166,2025-04-01
2,39.923,-75.182,2025-04-01
3,39.963,-75.174,2025-04-01
4,39.940,-75.214,2025-04-01


In [12]:
# Create a cache file to store all temp, wind_speed, and humidity data from API calls
CACHE_FILE = "weather_cache_q2.json"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r") as f:
        weather_cache = json.load(f)
else:
    weather_cache = {}

In [13]:
# Open a single session with OpenWeather API and define functions 
session = requests.Session()

def get_cache_key(lat, lon, date_str):
    return f"{lat}_{lon}_{date_str}"

def get_daily_weather(lat, lon, date_str):
    date_str = str(date_str)
    cache_key = get_cache_key(lat, lon, date_str)
    
    # Return cached result if already available
    if cache_key in weather_cache:
        return weather_cache[cache_key]

    params = {
        "lat": lat,
        "lon": lon,
        "date": date_str,
        "appid": API_KEY,
        "units": "imperial"
    }

    try:
        response = session.get(BASE_URL, params=params, timeout=30)

        if response.status_code != 200:
            result = {
                "start_lat": lat,
                "start_lon": lon,
                "start_date": date_str,
                "temp_c": None,
                "temp_f": None,
                "wind_speed": None,
                "humidity": None,
                "api_status": response.status_code
            }
        else:
            data = response.json()

            temp_f = data.get("main", {}).get("temp")
            temp_c = ((temp_f - 32) * 5 / 9) if temp_f is not None else None
            
            wind_speed = data.get("wind", {}).get("speed")

            humidity = data.get("main", {}).get("humidity")

            result = {
                "start_lat": lat,
                "start_lon": lon,
                "start_date": date_str,
                "temp_c": temp_c,
                "temp_f": temp_f,
                "wind_speed": wind_speed,
                "humidity": humidity,
                "api_status": response.status_code
            }

    except Exception as e:
        result = {
            "start_lat": lat,
            "start_lon": lon,
            "start_date": date_str,
            "temp_c": None,
            "temp_f": None,
            "wind_speed": None,
            "humidity": None,
            "api_status": f"error: {e}"
        }

    # Save to cache in memory
    weather_cache[cache_key] = result
    return result

In [14]:
# Create list containing all data from OpenWeather API calls from cache file to convert to a dataframe
weather_records = []

for i, row in weather_keys.iterrows():
    weather_row = get_daily_weather(
        lat=row["start_lat"],
        lon=row["start_lon"],
        date_str=row["start_date"]
    )
    weather_records.append(weather_row)

    # Save cache every 100 calls
    if i % 100 == 0:
        with open(CACHE_FILE, "w") as f:
            json.dump(weather_cache, f)
            print(f"Cache saved. Set - {i}")

    time.sleep(0.1)  # Small pause to avoid rate limiting

# Final save
with open(CACHE_FILE, "w") as f:
    json.dump(weather_cache, f)

weather_df = pd.DataFrame(weather_records)
weather_df.head()

Cache saved. Set - 0
Cache saved. Set - 100
Cache saved. Set - 200
Cache saved. Set - 300
Cache saved. Set - 400
Cache saved. Set - 500
Cache saved. Set - 600
Cache saved. Set - 700
Cache saved. Set - 800
Cache saved. Set - 900
Cache saved. Set - 1000
Cache saved. Set - 1100
Cache saved. Set - 1200
Cache saved. Set - 1300
Cache saved. Set - 1400
Cache saved. Set - 1500
Cache saved. Set - 1600
Cache saved. Set - 1700
Cache saved. Set - 1800
Cache saved. Set - 1900
Cache saved. Set - 2000
Cache saved. Set - 2100
Cache saved. Set - 2200
Cache saved. Set - 2300
Cache saved. Set - 2400
Cache saved. Set - 2500
Cache saved. Set - 2600
Cache saved. Set - 2700
Cache saved. Set - 2800
Cache saved. Set - 2900
Cache saved. Set - 3000
Cache saved. Set - 3100
Cache saved. Set - 3200
Cache saved. Set - 3300
Cache saved. Set - 3400
Cache saved. Set - 3500
Cache saved. Set - 3600
Cache saved. Set - 3700
Cache saved. Set - 3800
Cache saved. Set - 3900
Cache saved. Set - 4000
Cache saved. Set - 4100
Cach

,start_lat,start_lon,start_date,temp_c,temp_f,wind_speed,humidity,api_status
0,39.955,-75.183,2025-04-01,22.311111,72.16,7.16,60.0,200
1,39.963,-75.166,2025-04-01,22.338889,72.21,7.11,60.0,200
2,39.923,-75.182,2025-04-01,22.477778,72.46,7.23,60.0,200
3,39.963,-75.174,2025-04-01,22.338889,72.21,7.11,60.0,200
4,39.940,-75.214,2025-04-01,22.372222,72.27,7.25,60.0,200


### Merge Indego and Weather dataframes

In [15]:
# --- Normalize indego_df keys ---
indego_df["start_date"] = pd.to_datetime(indego_df["start_date"], errors="coerce").dt.strftime("%Y-%m-%d")
indego_df["start_lat_round"] = pd.to_numeric(indego_df["start_lat_round"], errors="coerce").round(3)
indego_df["start_lon_round"] = pd.to_numeric(indego_df["start_lon_round"], errors="coerce").round(3)

# --- Normalize weather_df keys ---
weather_df["start_date"] = pd.to_datetime(weather_df["start_date"], errors="coerce").dt.strftime("%Y-%m-%d")
weather_df["start_lat"] = pd.to_numeric(weather_df["start_lat"], errors="coerce").round(3)
weather_df["start_lon"] = pd.to_numeric(weather_df["start_lon"], errors="coerce").round(3)


In [16]:
weather_df.head()

,start_lat,start_lon,start_date,temp_c,temp_f,wind_speed,humidity,api_status
0,39.955,-75.183,2025-04-01,22.311111,72.16,7.16,60.0,200
1,39.963,-75.166,2025-04-01,22.338889,72.21,7.11,60.0,200
2,39.923,-75.182,2025-04-01,22.477778,72.46,7.23,60.0,200
3,39.963,-75.174,2025-04-01,22.338889,72.21,7.11,60.0,200
4,39.940,-75.214,2025-04-01,22.372222,72.27,7.25,60.0,200


In [17]:
# Merge weather_df with indego_df
indego_weather_df = indego_df.merge(
    weather_df,
    left_on=["start_lat_round", "start_lon_round", "start_date"],
    right_on=["start_lat", "start_lon", "start_date"],
    how="left"
)

# Validate table merge
indego_weather_df.head(10)

,trip_id,duration,start_station,start_lat_x,start_lon_x,end_station,end_lat,end_lon,bike_id,plan_duration,...,end_time_only,start_lat_round,start_lon_round,start_lat_y,start_lon_y,temp_c,temp_f,wind_speed,humidity,api_status
0,1164246461,11,3022,39.954720,-75.183228,3064,39.938400,-75.173271,31751,30,...,00:15:00,39.955,-75.183,39.955,-75.183,22.311111,72.16,7.16,60.0,200
1,1164246634,31,3040,39.962891,-75.166061,3100,39.927769,-75.151031,14481,30,...,00:35:00,39.963,-75.166,39.963,-75.166,22.338889,72.21,7.11,60.0,200
2,1164246553,9,3396,39.923271,-75.182098,3349,39.936508,-75.186211,02724,30,...,00:26:00,39.923,-75.182,39.923,-75.182,22.477778,72.46,7.23,60.0,200
3,1164246521,3,3054,39.962502,-75.174202,3235,39.959999,-75.165100,24841,365,...,00:23:00,39.963,-75.174,39.963,-75.174,22.338889,72.21,7.11,60.0,200
4,1164246659,11,3280,39.939678,-75.213623,3349,39.936508,-75.186211,25744,30,...,00:36:00,39.940,-75.214,39.940,-75.214,22.372222,72.27,7.25,60.0,200
5,1164246764,7,3301,39.950291,-75.171257,3051,39.967442,-75.175072,31234,365,...,00:47:00,39.950,-75.171,39.950,-75.171,22.338889,72.21,7.11,60.0,200
6,1164246881,13,3158,39.925522,-75.169037,3028,39.940609,-75.149582,16381,1,...,01:26:00,39.926,-75.169,39.926,-75.169,22.450000,72.41,7.18,60.0,200
7,1164246912,12,3374,39.972801,-75.179710,3154,39.959240,-75.158211,22105,30,...,01:33:00,39.973,-75.180,39.973,-75.180,22.411111,72.34,7.09,60.0,200
8,1164250593,3,3371,39.953400,-75.154297,3378,39.952381,-75.147278,23237,30,...,02:02:00,39.953,-75.154,39.953,-75.154,22.338889,72.21,7.11,60.0,200
9,1164258007,58,3271,39.947601,-75.229462,3344,39.959610,-75.233543,05376,30,...,04:02:00,39.948,-75.229,39.948,-75.229,22.272222,72.09,7.18,61.0,200


In [18]:
indego_weather_df.tail(10)

,trip_id,duration,start_station,start_lat_x,start_lon_x,end_station,end_lat,end_lon,bike_id,plan_duration,...,end_time_only,start_lat_round,start_lon_round,start_lat_y,start_lon_y,temp_c,temp_f,wind_speed,humidity,api_status
365750,1203068619,7,3279,39.926498,-75.166801,3098,39.934311,-75.160423,03529,30,...,23:36:00,39.926,-75.167,39.926,-75.167,23.800000,74.84,7.20,55.0,200
365751,1203068801,7,3203,39.940769,-75.172272,3007,39.945171,-75.159927,28155,365,...,23:39:00,39.941,-75.172,39.941,-75.172,23.800000,74.84,7.20,55.0,200
365752,1203069034,13,3104,39.966640,-75.192093,3104,39.966640,-75.192093,05196,30,...,23:45:00,39.967,-75.192,39.967,-75.192,23.661111,74.59,7.20,55.0,200
365753,1203069096,11,3353,40.004959,-75.152328,3017,39.980030,-75.143707,23450,30,...,23:46:00,40.005,-75.152,40.005,-75.152,23.372222,74.07,7.07,55.0,200
365754,1203069064,8,3033,39.950050,-75.156723,3387,39.970119,-75.146660,31731,365,...,23:45:00,39.950,-75.157,39.950,-75.157,23.677778,74.62,7.16,55.0,200
365755,1203070684,24,3255,39.950951,-75.164383,3255,39.950951,-75.164383,02604,1,...,00:01:00,39.951,-75.164,39.951,-75.164,23.677778,74.62,7.16,55.0,200
365756,1203069362,6,3058,39.967159,-75.170013,3168,39.951340,-75.173943,29560,30,...,23:56:00,39.967,-75.170,39.967,-75.170,23.638889,74.55,7.11,55.0,200
365757,1203069342,5,3009,39.955761,-75.189819,3035,39.962711,-75.194191,02680,30,...,23:56:00,39.956,-75.190,39.956,-75.190,23.661111,74.59,7.20,55.0,200
365758,1203109583,437,3357,39.910400,-75.165878,3068,39.935490,-75.167107,18769,30,...,07:11:00,39.910,-75.166,39.910,-75.166,23.850000,74.93,7.27,55.0,200
365759,1203073458,9,3086,39.940189,-75.166908,3163,39.949741,-75.180969,29485,30,...,00:06:00,39.940,-75.167,39.940,-75.167,23.800000,74.84,7.20,55.0,200


In [19]:
# Save merged table to a CSV file
indego_weather_df.to_csv('indego_weather_merged_table.csv', index=False)

In [20]:
# Examine Indego-Weather merged table
indego_weather_df[["temp_f","temp_c", "wind_speed", "humidity"]].describe()

,temp_f,temp_c,wind_speed,humidity
count,365685.000000,365685.000000,365685.000000,365685.000000
mean,72.977206,22.765114,7.139082,60.421499
std,0.655572,0.364207,0.053059,2.453961
min,71.260000,21.811111,7.000000,55.000000
25%,72.610000,22.561111,7.110000,59.000000
50%,72.820000,22.677778,7.160000,60.000000
75%,73.150000,22.861111,7.180000,63.000000
max,74.980000,23.877778,9.220000,64.000000


In [21]:
indego_weather_df.isna().sum()

trip_id                 0
duration                0
start_station           0
start_lat_x             0
start_lon_x             0
end_station             0
end_lat                 0
end_lon                 0
bike_id                 0
plan_duration           0
trip_route_category     0
passholder_type         4
bike_type               0
start_date              0
start_time_only         0
end_date                0
end_time_only           0
start_lat_round         0
start_lon_round         0
start_lat_y             0
start_lon_y             0
temp_c                 75
temp_f                 75
wind_speed             75
humidity               75
api_status              0
dtype: int64

In [23]:
# Read the CSV file to dataframe again
indego_weather_csv_df = pd.read_csv("indego_weather_merged_table.csv", low_memory=False)
indego_weather_csv_df.head()

,trip_id,duration,start_station,start_lat_x,start_lon_x,end_station,end_lat,end_lon,bike_id,plan_duration,...,end_time_only,start_lat_round,start_lon_round,start_lat_y,start_lon_y,temp_c,temp_f,wind_speed,humidity,api_status
0,1164246461,11,3022,39.954720,-75.183228,3064,39.938400,-75.173271,31751,30,...,00:15:00,39.955,-75.183,39.955,-75.183,22.311111,72.16,7.16,60.0,200
1,1164246634,31,3040,39.962891,-75.166061,3100,39.927769,-75.151031,14481,30,...,00:35:00,39.963,-75.166,39.963,-75.166,22.338889,72.21,7.11,60.0,200
2,1164246553,9,3396,39.923271,-75.182098,3349,39.936508,-75.186211,02724,30,...,00:26:00,39.923,-75.182,39.923,-75.182,22.477778,72.46,7.23,60.0,200
3,1164246521,3,3054,39.962502,-75.174202,3235,39.959999,-75.165100,24841,365,...,00:23:00,39.963,-75.174,39.963,-75.174,22.338889,72.21,7.11,60.0,200
4,1164246659,11,3280,39.939678,-75.213623,3349,39.936508,-75.186211,25744,30,...,00:36:00,39.940,-75.214,39.940,-75.214,22.372222,72.27,7.25,60.0,200


## Upload Merged Indego-Weather dataframe to MongoDB

In [24]:
import numpy as np

# Select database and collection
db = db_connections.mogodb_client["Final_Project"]
collection = db["indego_weather"]

# Copy dataframe
mongo_df = indego_weather_csv_df.copy()

# Replace NaN with None
mongo_df = mongo_df.replace({np.nan: None})

# Convert datetime-like columns to string
for col in mongo_df.columns:
    if str(mongo_df[col].dtype).startswith("datetime"):
        mongo_df[col] = mongo_df[col].astype(str)

# Convert to list of dictionaries
records = mongo_df.to_dict(orient="records")

# Delete Colletion first before re-inserting data
# collection.delete_many({})

# Insert into MongoDB
insert_result = collection.insert_many(records)

print(f"Inserted {len(insert_result.inserted_ids)} documents into MongoDB.")

Inserted 365760 documents into MongoDB.
